In [1]:
import tensorflow as tf

path = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path, 'rb').read().decode(encoding='utf-8')
print(text[:500])  

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [3]:
import numpy as np
vocab = sorted(set(text))          # unique characters
print(f'Total unique characters: {len(vocab)}')

# Character → Number
char2idx = {char: idx for idx, char in enumerate(vocab)}

# Number → Character (wapas text banane ke liye)
idx2char = np.array(vocab)

# Poora text numbers mein
text_as_int = np.array([char2idx[c] for c in text])

Total unique characters: 65


In [4]:
seq_length = 100
examples_per_epoch = len(text) // (seq_length + 1)

char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]   # pehle 100 characters = INPUT
    target_text = chunk[1:]   # agle 100 characters = ANSWER
    return input_text, target_text

dataset = sequences.map(split_input_target)
dataset = dataset.shuffle(10000).batch(64, drop_remainder=True)

In [6]:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 512

# ---- Simple RNN ----
rnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(None,), batch_size=64),
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.SimpleRNN(rnn_units, return_sequences=False),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

# ---- LSTM ----
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(None,), batch_size=64),
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.LSTM(rnn_units, return_sequences=False),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

# ---- GRU ----
gru_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(None,), batch_size=64),
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.GRU(rnn_units, return_sequences=False),
    tf.keras.layers.Dense(vocab_size, activation='softmax')
])

In [7]:
def compile_and_train(model, name):
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(dataset, epochs=10)
    return history

rnn_history  = compile_and_train(rnn_model,  "RNN")
lstm_history = compile_and_train(lstm_model, "LSTM")
gru_history  = compile_and_train(gru_model,  "GRU")

Epoch 1/10


ValueError: Argument `output` must have rank (ndim) `target.ndim - 1`. Received: target.shape=(64, 100), output.shape=(64, 65)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Loss
plt.subplot(1, 2, 1)
plt.plot(rnn_history.history['loss'],  label='RNN')
plt.plot(lstm_history.history['loss'], label='LSTM')
plt.plot(gru_history.history['loss'],  label='GRU')
plt.title('Training Loss')
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(rnn_history.history['accuracy'],  label='RNN')
plt.plot(lstm_history.history['accuracy'], label='LSTM')
plt.plot(gru_history.history['accuracy'],  label='GRU')
plt.title('Training Accuracy')
plt.legend()

plt.show()

In [ ]:
def generate_text(model, start_string, num_generate=200):
    # Start string ko numbers mein badlo
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    generated = []
    model.reset_states()

    for _ in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        # Random sampling — temperature add karo variety ke liye
        predicted_id = tf.random.categorical(
            tf.math.log(predictions) / 0.5,  # 0.5 = temperature
            num_samples=1
        )[-1, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        generated.append(idx2char[predicted_id])

    return start_string + ''.join(generated)

print("RNN:\n",  generate_text(rnn_model,  "ROMEO: "))
print("LSTM:\n", generate_text(lstm_model, "ROMEO: "))
print("GRU:\n",  generate_text(gru_model,  "ROMEO: "))